## ALS

In [1]:
from implicit.als import AlternatingLeastSquares
import polars as pl
from datetime import date, timedelta

In [11]:
user_actions_full = pl.read_parquet('../data/user_actions_full')

In [39]:
user_actions_full

user_id,product_id,date,action_type
i32,i64,date,str
10542465,166798937,2024-04-03,"""to_cart"""
8348185,146763544,2024-07-22,"""order"""
1423743,180730425,2024-05-11,"""to_cart"""
2102996,322819992,2024-07-29,"""click"""
2569829,25343359,2024-06-05,"""order"""
…,…,…,…
10655608,172793428,2024-04-09,"""click"""
8142373,316742407,2024-05-23,"""click"""
5884019,1215921361,2024-07-05,"""order"""


In [4]:
TEST_START = date(2024, 7, 1)

In [6]:
train_orders = (
    user_actions_full
    .filter(pl.col('date') < TEST_START)
    .filter(pl.col('date') >= TEST_START - timedelta(days=3 * 30))
    .filter(pl.col('action_type') == 'order')
    .select('user_id', 'product_id', 'date')
)

In [2]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"

In [8]:
from rectools.models import ImplicitALSWrapperModel
from rectools.dataset import Dataset

In [15]:
train_dataset = Dataset.construct(
    interactions_df=(
        train_orders
        .select('user_id', pl.col('product_id').alias('item_id'), pl.lit(1).alias('weight'), pl.col('date').alias('datetime'))
        .to_pandas()
    ),
)

In [16]:
model = ImplicitALSWrapperModel(
    AlternatingLeastSquares(
        factors=64,
        regularization=0.01,
        alpha=1.0,
        random_state=0,
        use_gpu=False,
        num_threads=32,
        iterations=15,
    ),
)

In [17]:
%%time
model.fit(train_dataset)

CPU times: user 21min 20s, sys: 9min 20s, total: 30min 40s
Wall time: 4min 40s


In [18]:
TEST_START - timedelta(days=3 * 30), TEST_START - timedelta(days=1)

(datetime.date(2024, 4, 2), datetime.date(2024, 6, 30))

In [19]:
model.save('../models/implicit_als_0402_0630.pkl')

248753186

In [21]:
from rectools.models import load_model

In [22]:
model_load = load_model('../models/implicit_als_0402_0630.pkl')

In [23]:
recoms = model_load.recommend(
    users=[7829309],
    dataset=train_dataset,
    k=100,
    filter_viewed=False,
)

In [14]:
recoms

,user_id,item_id,score,rank
0,7829309,145923184,1.063576,1
1,7829309,148481523,1.055408,2
2,7829309,261375234,0.932172,3
3,7829309,267896010,0.655970,4
4,7829309,267897402,0.636042,5
...,...,...,...,...
95,7829309,546521032,0.152953,96
96,7829309,267896691,0.152602,97
97,7829309,267896015,0.152581,98
98,7829309,291634789,0.151809,99


In [27]:
test_orders = (
    user_actions_full
    .filter(pl.col('date') >= TEST_START)
    .filter(pl.col('action_type') == 'order')
    .select('user_id', 'product_id')
)
sample_users = (
    test_orders
    .group_by('user_id')
    .agg(
        pl.col("product_id").unique().alias("ids")
    )
    .sample(n=1000, seed=0)
)

In [35]:
model.recommend(
    users=sample_df,
    dataset=train_dataset,
    k=30,
    filter_viewed=False,
    on_unsupported_targets="ignore"
)

,user_id,item_id,score,rank
0,9201201,142120588,0.946217,1
1,9201201,184276495,0.606800,2
2,9201201,145803938,0.369289,3
3,9201201,138860233,0.362406,4
4,9201201,148234014,0.359161,5
...,...,...,...,...
15685,1375307,261375240,0.046411,26
15686,1375307,621041182,0.045993,27
15687,1375307,686790318,0.045402,28
15688,1375307,690589606,0.045336,29
